In [1]:
# =========== IMPORT LIBRARY ===========
import numpy as np
import pandas as pd
import random
import plotly.graph_objects as go
from collections import defaultdict

In [3]:
# =========== KONFIGURASI DATA ===========
class Container:
    def __init__(self, length=6, width=5, height=5, max_weight=50):
        self.length = length
        self.width = width
        self.height = height
        self.max_weight = max_weight
        self.volume = length * width * height
        # Batas pusat massa (dalam satuan panjang dari tengah)
        self.cog_limit_x = length * 0.1  # 10% dari panjang
        self.cog_limit_y = width * 0.1   # 10% dari lebar
        self.cog_limit_z = height * 0.15 # 15% dari tinggi (lebih toleran)

class Package:
    def __init__(self, id_, length, width, height, weight):
        self.id = id_
        self.original = (length, width, height)
        self.weight = weight
        self.volume = length * width * height
        self.orientations = self.generate_orientations()

    def generate_orientations(self):
        l, w, h = self.original
        return {
            1: (l, w, h),   # L,W,H
            2: (l, h, w),   # L,H,W
            3: (w, l, h),   # W,L,H
            4: (w, h, l),   # W,H,L
            5: (h, l, w),   # H,L,W
            6: (h, w, l)    # H,W,L
        }

def bottom_left_fill_with_fitness(chromosome, container, packages_dict):
    """
    Menempatkan paket dengan algoritma Bottom-Left-Fill
    dan langsung menghitung fitness
    """
    positions = []
    placed = []
    space_grid = np.zeros((container.length, container.width, container.height), dtype=int)

    # Untuk tracking pusat massa
    total_mass = 0
    total_mass_x = 0
    total_mass_y = 0
    total_mass_z = 0
    total_volume = 0

    # Flag untuk B5
    all_stability_valid = True
    total_weight_placed = 0

    for gene in chromosome:
        p_id = gene[0]
        orientation = gene[1]
        package = packages_dict[p_id]
        dims = package.orientations[orientation]
        placed_flag = False

        # Cari semua posisi yang mungkin (z terendah dulu, lalu y, lalu x)
        for z in range(container.height - dims[2] + 1):
            for y in range(container.width - dims[1] + 1):
                for x in range(container.length - dims[0] + 1):
                    # Cek apakah space kosong
                    if np.all(space_grid[x:x+dims[0], y:y+dims[1], z:z+dims[2]] == 0):

                        # =========== CEK STABILITAS (B5) ===========
                        stability_valid = True
                        if z > 0:  # Tidak di lantai dasar
                            support_area = 0
                            total_area = dims[0] * dims[1]

                            # Hitung area yang ditopang oleh kargo di bawahnya
                            for xp in range(x, x + dims[0]):
                                for yp in range(y, y + dims[1]):
                                    if space_grid[xp, yp, z-1] != 0:
                                        support_area += 1

                            # Harus minimal 50% area tertopang
                            if support_area < 0.5 * total_area:
                                stability_valid = False
                                all_stability_valid = False

                        if not stability_valid:
                            continue  # Lewati posisi ini

                        # =========== TEMPATKAN PAKET ===========
                        space_grid[x:x+dims[0], y:y+dims[1], z:z+dims[2]] = int(p_id[1:])

                        # Hitung pusat massa kargo ini
                        cog_x = x + dims[0] / 2.0
                        cog_y = y + dims[1] / 2.0
                        cog_z = z + dims[2] / 2.0

                        # Update total pusat massa
                        total_mass += package.weight
                        total_mass_x += package.weight * cog_x
                        total_mass_y += package.weight * cog_y
                        total_mass_z += package.weight * cog_z

                        # Update total volume dan berat
                        volume = dims[0] * dims[1] * dims[2]
                        total_volume += volume
                        total_weight_placed += package.weight

                        positions.append({
                            'id': p_id,
                            'x': x, 'y': y, 'z': z,
                            'dx': dims[0], 'dy': dims[1], 'dz': dims[2],
                            'weight': package.weight,
                            'volume': volume,
                            'orientation': orientation,
                            'cog_x': cog_x,
                            'cog_y': cog_y,
                            'cog_z': cog_z,
                            'placed': True
                        })
                        placed.append(p_id)
                        placed_flag = True
                        break
                if placed_flag:
                    break
            if placed_flag:
                break

        if not placed_flag:
            # Tidak bisa ditempatkan
            positions.append({
                'id': p_id,
                'x': -1, 'y': -1, 'z': -1,
                'dx': dims[0], 'dy': dims[1], 'dz': dims[2],
                'weight': package.weight,
                'volume': dims[0] * dims[1] * dims[2],
                'orientation': orientation,
                'placed': False
            })

    # =========== CEK KAPASITAS BEBAN TOTAL (B4) ===========
    B4 = 1 if total_weight_placed <= container.max_weight else 0

    # B5 = 1 jika semua paket stabil, 0 jika ada yang tidak stabil
    B5 = 1 if all_stability_valid else 0

    # Hitung pusat massa total
    if total_mass > 0:
        cog_total_x = total_mass_x / total_mass
        cog_total_y = total_mass_y / total_mass
        cog_total_z = total_mass_z / total_mass
    else:
        cog_total_x = cog_total_y = cog_total_z = 0

    # =========== HITUNG PENALTI PUSAT MASSA (B1, B2, B3) ===========
    container_center_x = container.length / 2.0
    container_center_y = container.width / 2.0
    container_center_z = container.height / 2.0

    dev_x = abs(cog_total_x - container_center_x)
    dev_y = abs(cog_total_y - container_center_y)
    dev_z = abs(cog_total_z - container_center_z)

    B1 = max(0, dev_x - container.cog_limit_x)
    B2 = max(0, dev_y - container.cog_limit_y)
    B3 = max(0, dev_z - container.cog_limit_z)

    penalty_cog = B1 + B2 + B3

    # =========== HITUNG FITNESS AKHIR ===========
    fitness_raw = total_volume - penalty_cog
    fitness_final = fitness_raw * B4 * B5

    # Volume utilization
    volume_utilization = (total_volume / container.volume) * 100 if container.volume > 0 else 0

    # Metadata untuk analisis
    metadata = {
        'fitness': fitness_final,
        'volume_utilization': volume_utilization,
        'total_volume': total_volume,
        'total_weight': total_weight_placed,
        'penalty_cog': penalty_cog,
        'B1': B1, 'B2': B2, 'B3': B3,
        'B4': B4, 'B5': B5,
        'num_placed': len(placed),
        'positions': positions,
        'cog_actual': (cog_total_x, cog_total_y, cog_total_z),
        'cog_deviation': (dev_x, dev_y, dev_z)
    }

    return fitness_final, metadata

# =========== FUNGSI GENETIC ALGORITHM ===========
def create_chromosome(packages_list):
    """Membuat kromosom acak"""
    ids = [p.id for p in packages_list]
    random.shuffle(ids)
    chromosome = []
    for p_id in ids:
        orientation = random.randint(1, 6)
        chromosome.append((p_id, orientation))
    return chromosome

def tournament_selection(population, fitness_scores, tournament_size=3):
    """Seleksi dengan metode tournament (mengembalikan INDEX)"""
    indices = random.sample(range(len(population)), tournament_size)
    best_idx = max(indices, key=lambda i: fitness_scores[i])
    return best_idx

def pmx_crossover(parent1, parent2):
    """Partially Matched Crossover"""
    size = len(parent1)
    p1 = parent1.copy()
    p2 = parent2.copy()

    # Pilih dua titik potong acak
    cut1 = random.randint(0, size-2)
    cut2 = random.randint(cut1+1, size-1)

    child1 = [None] * size
    child2 = [None] * size

    # Salin bagian tengah
    child1[cut1:cut2] = p1[cut1:cut2]
    child2[cut1:cut2] = p2[cut1:cut2]

    # Mapping
    mapping1 = {}
    mapping2 = {}
    for i in range(cut1, cut2):
        mapping1[p1[i][0]] = p2[i][0]
        mapping2[p2[i][0]] = p1[i][0]

    # Isi sisa posisi untuk child1
    for i in range(size):
        if i < cut1 or i >= cut2:
            gene = p2[i]
            while gene[0] in [g[0] for g in child1 if g is not None]:
                if gene[0] in mapping1:
                    gene_id = mapping1[gene[0]]
                    for g in p2:
                        if g[0] == gene_id:
                            gene = g
                            break
                else:
                    for g in p1:
                        if g[0] not in [cg[0] for cg in child1 if cg is not None]:
                            gene = g
                            break
            child1[i] = gene

    # Isi sisa posisi untuk child2
    for i in range(size):
        if i < cut1 or i >= cut2:
            gene = p1[i]
            while gene[0] in [g[0] for g in child2 if g is not None]:
                if gene[0] in mapping2:
                    gene_id = mapping2[gene[0]]
                    for g in p1:
                        if g[0] == gene_id:
                            gene = g
                            break
                else:
                    for g in p2:
                        if g[0] not in [cg[0] for cg in child2 if cg is not None]:
                            gene = g
                            break
            child2[i] = gene

    return child1, child2

def mutate(chromosome):
    """
    Mutasi kromosom dengan dua kemungkinan:
    - 50%: Swap urutan (menukar posisi dua paket)
    - 50%: Swap rotasi (mengubah orientasi satu paket)
    """
    mutated = chromosome.copy()
    
    # Pilih jenis mutasi: 0 = swap urutan, 1 = swap rotasi
    mutation_type = random.choice(['swap_order', 'swap_rotation'])
    
    if mutation_type == 'swap_order':
        # Swap urutan: tukar posisi dua paket yang berbeda
        if len(mutated) >= 2:
            # Pilih dua indeks berbeda secara acak
            idx1, idx2 = random.sample(range(len(mutated)), 2)
            # Tukar posisi kedua paket
            mutated[idx1], mutated[idx2] = mutated[idx2], mutated[idx1]
    else:
        # Swap rotasi: ubah orientasi satu paket secara acak
        idx = random.randint(0, len(mutated)-1)
        old_orient = mutated[idx][1]
        new_orient = random.randint(1, 6)
        # Pastikan orientasi berubah (opsional)
        while new_orient == old_orient and len(mutated) > 0:
            new_orient = random.randint(1, 6)
        mutated[idx] = (mutated[idx][0], new_orient)
    
    return mutated

# =========== VISUALISASI 3D ===========
def visualize_packing(positions, container_dims, title="3D Bin Packing"):
    """Visualisasi terbaik dengan kubus transparan dan garis tepi"""
    fig = go.Figure()

    # Kontainer sebagai wireframe transparan
    container_lines = [
        [0, 1], [1, 2], [2, 3], [3, 0],  # Bottom square
        [4, 5], [5, 6], [6, 7], [7, 4],  # Top square
        [0, 4], [1, 5], [2, 6], [3, 7]   # Vertical edges
    ]

    container_vertices = [
        [0, 0, 0],
        [container_dims[0], 0, 0],
        [container_dims[0], container_dims[1], 0],
        [0, container_dims[1], 0],
        [0, 0, container_dims[2]],
        [container_dims[0], 0, container_dims[2]],
        [container_dims[0], container_dims[1], container_dims[2]],
        [0, container_dims[1], container_dims[2]]
    ]

    for line in container_lines:
        fig.add_trace(go.Scatter3d(
            x=[container_vertices[line[0]][0], container_vertices[line[1]][0]],
            y=[container_vertices[line[0]][1], container_vertices[line[1]][1]],
            z=[container_vertices[line[0]][2], container_vertices[line[1]][2]],
            mode='lines',
            line=dict(color='gray', width=2, dash='dash'),
            name='Container',
            showlegend=True if line == [0, 1] else False
        ))

    # Warna untuk paket
    colors = ['rgba(255, 0, 0, 0.7)', 'rgba(0, 255, 0, 0.7)', 'rgba(0, 0, 255, 0.7)',
              'rgba(255, 165, 0, 0.7)', 'rgba(128, 0, 128, 0.7)', 'rgba(255, 255, 0, 0.7)']

    for i, pos in enumerate(positions):
        if pos['x'] != -1:
            x, y, z = pos['x'], pos['y'], pos['z']
            dx, dy, dz = pos['dx'], pos['dy'], pos['dz']

            color_idx = int(pos['id'][1:]) - 1 if pos['id'][1:].isdigit() else i
            color = colors[color_idx % len(colors)]

            fig.add_trace(go.Mesh3d(
                x=[x, x+dx, x+dx, x, x, x+dx, x+dx, x],
                y=[y, y, y+dy, y+dy, y, y, y+dy, y+dy],
                z=[z, z, z, z, z+dz, z+dz, z+dz, z+dz],
                i=[0, 0, 0, 1, 1, 2],
                j=[1, 2, 4, 3, 5, 6],
                k=[2, 3, 5, 7, 6, 7],
                color=color,
                opacity=0.5,
                flatshading=True,
                name=f"{pos['id']} ({dx}×{dy}×{dz})"
            ))

    fig.update_layout(
        title=dict(text=title, x=0.5, xanchor='center'),
        scene=dict(
            xaxis_title='Panjang (X)',
            yaxis_title='Lebar (Y)',
            zaxis_title='Tinggi (Z)',
            aspectmode='data',
            camera=dict(
                up=dict(x=0, y=0, z=1),
                center=dict(x=0, y=0, z=0),
                eye=dict(x=1.5, y=1.5, z=1.5)
            )
        ),
        showlegend=True,
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
    )

    return fig

# =========== ALGORITMA GENETIK UTAMA ===========
class GeneticAlgorithm:
    def __init__(self, population_size, crossover_rate=0.8, mutation_rate=0.2):
        """
        Args:
            population_size: Ukuran populasi
            crossover_rate: Probabilitas crossover terjadi (0-1)
            mutation_rate: Probabilitas mutasi terjadi (0-1)
        """
        self.population_size = population_size
        self.crossover_rate = crossover_rate
        self.mutation_rate = mutation_rate

    def generate_offspring_with_details(self, population, fitness_scores):
        """
        Menghasilkan offspring sebanyak population_size dengan detail proses
        - Crossover: dari 2 parent → 2 child (dengan 2 info terpisah)
        - Mutasi: dari 1 parent → 1 child (dengan 1 info)
        """
        offspring = []
        offspring_info = []
        crossover_details = []
        mutation_details = []

        print(f"\n  {'='*50}")
        print(f"  PROSES GENERASI OFFSPRING")
        print(f"  {'='*50}")
        print(f"  Target: {self.population_size} offspring")
        print(f"  Crossover rate: {self.crossover_rate*100:.1f}%")
        print(f"  Mutation rate: {self.mutation_rate*100:.1f}%")
        print(f"  {'='*50}")

        # =========== PERBAIKAN: Hitung alokasi di awal ===========
        # Target jumlah offspring dari crossover dan mutasi
        target_crossover_children = int(self.population_size * self.crossover_rate)
        target_mutation_children = self.population_size - target_crossover_children
        
        # Pastikan genap untuk crossover (karena 1 crossover = 2 child)
        if target_crossover_children % 2 != 0:
            target_crossover_children -= 1
            target_mutation_children += 1
        
        target_crossover_ops = target_crossover_children // 2
        target_mutation_ops = target_mutation_children
        
        print(f"  Target alokasi:")
        print(f"  - Crossover: {target_crossover_ops} operasi → {target_crossover_children} child")
        print(f"  - Mutasi: {target_mutation_ops} operasi → {target_mutation_children} child")
        print(f"  {'='*50}")
        
        crossover_count = 0
        mutation_count = 0
        
        # =========== PRODUKSI CROSSOVER ===========
        for _ in range(target_crossover_ops):
            # Seleksi 2 parent untuk crossover
            parent1_idx = tournament_selection(population, fitness_scores)
            parent2_idx = tournament_selection(population, fitness_scores)
            parent1 = population[parent1_idx]
            parent2 = population[parent2_idx]
            
            # Lakukan crossover
            child1, child2 = pmx_crossover(parent1, parent2)
            crossover_count += 1
            
            # Simpan detail crossover
            crossover_detail = {
                'parents': (parent1_idx, parent2_idx),
                'parent1_fitness': fitness_scores[parent1_idx],
                'parent2_fitness': fitness_scores[parent2_idx],
                'children': (child1.copy(), child2.copy())
            }
            crossover_details.append(crossover_detail)
            
            # Simpan informasi untuk CHILD 1 & 2
            for child_num in [1, 2]:
                offspring_info.append({
                    'type': 'crossover',
                    'parents': (parent1_idx, parent2_idx),
                    'child_num': child_num,
                    'parent1_fitness': fitness_scores[parent1_idx],
                    'parent2_fitness': fitness_scores[parent2_idx]
                })
            
            # Tambahkan ke offspring
            offspring.append(child1)
            offspring.append(child2)
            
            print(f"\n  >> CROSSOVER #{crossover_count}:")
            print(f"     Parent 1 (K{parent1_idx+1}): {parent1[:3]}... (fit: {fitness_scores[parent1_idx]:.2f})")
            print(f"     Parent 2 (K{parent2_idx+1}): {parent2[:3]}... (fit: {fitness_scores[parent2_idx]:.2f})")
            print(f"     → Child 1: {child1[:3]}...")
            print(f"     → Child 2: {child2[:3]}...")
        
        # =========== PRODUKSI MUTASI ===========
        for _ in range(target_mutation_ops):
            # Seleksi 1 parent untuk mutasi
            parent_idx = tournament_selection(population, fitness_scores)
            parent = population[parent_idx]
            
            # Lakukan mutasi
            mutated = mutate(parent)
            mutation_count += 1
            
            # Simpan detail mutasi
            mutation_detail = {
                'parent': parent_idx,
                'parent_fitness': fitness_scores[parent_idx],
                'child': mutated.copy()
            }
            
            # Cari posisi yang dimutasi untuk detail
            for pos in range(len(parent)):
                if parent[pos] != mutated[pos]:
                    mutation_detail['mutation_position'] = pos
                    mutation_detail['original'] = parent[pos]
                    mutation_detail['mutated'] = mutated[pos]
                    break
            
            mutation_details.append(mutation_detail)
            
            # Simpan informasi
            offspring_info.append({
                'type': 'mutation',
                'parent': parent_idx,
                'parent_fitness': fitness_scores[parent_idx]
            })
            
            # Tambahkan ke offspring
            offspring.append(mutated)
            
            print(f"\n  >> MUTASI #{mutation_count}:")
            print(f"     Parent (K{parent_idx+1}): {parent[:3]}... (fit: {fitness_scores[parent_idx]:.2f})")
            
            # Cari posisi yang dimutasi
            for pos in range(len(parent)):
                if parent[pos] != mutated[pos]:
                    print(f"     → Child: {mutated[:3]}... (mutasi di posisi {pos+1}: {parent[pos]} → {mutated[pos]})")
                    break
        
        # Pastikan jumlah offspring sesuai target
        offspring = offspring[:self.population_size]
        offspring_info = offspring_info[:self.population_size]

        print(f"\n  {'='*50}")
        print(f"  RINGKASAN OFFSPRING:")
        print(f"  - Total crossover operations: {crossover_count} → {crossover_count*2} child")
        print(f"  - Total mutation operations: {mutation_count} → {mutation_count} child")
        print(f"  - Total offspring: {len(offspring)}")
        print(f"  {'='*50}")
        
        # Tampilkan semua offspring yang dihasilkan
        print(f"\n  DAFTAR SEMUA OFFSPRING (SEBELUM SELEKSI):")
        print(f"  {'-'*50}")
        for i, chrom in enumerate(offspring, 1):
            info = offspring_info[i-1]
            if info['type'] == 'crossover':
                print(f"  O{i}: [CROSSOVER] {chrom[:3]}... (dari K{info['parents'][0]+1} & K{info['parents'][1]+1})")
            else:
                print(f"  O{i}: [MUTASI] {chrom[:3]}... (dari K{info['parent']+1})")
        print(f"  {'-'*50}")
        
        return offspring, {
            'offspring_info': offspring_info,
            'crossover_details': crossover_details,
            'mutation_details': mutation_details
        }

    def select_best(self, combined_population, combined_fitness):
        """
        Memilih individu terbaik dari combined population (tanpa elitism)
        """
        # Urutkan indices berdasarkan fitness (descending)
        sorted_indices = np.argsort(combined_fitness)[::-1]

        # Ambil population_size terbaik
        best_indices = sorted_indices[:self.population_size]

        new_population = [combined_population[i] for i in best_indices]

        return new_population, best_indices

# =========== FUNGSI UNTUK ANALISIS DETAIL ===========
def analyze_chromosome(chromosome, container, packages_dict):
    """Analisis detail kromosom"""
    fitness, metadata = bottom_left_fill_with_fitness(chromosome, container, packages_dict)

    print(f"\n{'='*60}")
    print(f"ANALISIS DETAIL KROMOSOM")
    print(f"{'='*60}")
    print(f"Urutan: {chromosome}")

    print(f"\nPosisi akhir paket:")
    placed_count = 0
    for pos in metadata['positions']:
        if pos['placed']:
            placed_count += 1
            print(f"  {pos['id']}: Posisi({pos['x']},{pos['y']},{pos['z']}) "
                  f"Dimensi{pos['dx']}x{pos['dy']}x{pos['dz']} Berat:{pos['weight']}kg")
        else:
            print(f"  {pos['id']}: TIDAK TERPASANG Dimensi{pos['dx']}x{pos['dy']}x{pos['dz']}")

    print(f"\nPerhitungan Fitness (sesuai paper):")
    print(f"Total Volume Paket Terpasang (E): {metadata['total_volume']} cm³")
    print(f"Volume Kontainer: {container.volume} cm³")
    print(f"Volume Utilization: {metadata['volume_utilization']:.2f}%")
    print(f"Total Berat: {metadata['total_weight']} kg (Maks: {container.max_weight} kg)")
    print(f"Pusat Massa Aktual: ({metadata['cog_actual'][0]:.2f}, {metadata['cog_actual'][1]:.2f}, {metadata['cog_actual'][2]:.2f})")
    print(f"Deviasi Pusat Massa: ({metadata['cog_deviation'][0]:.2f}, {metadata['cog_deviation'][1]:.2f}, {metadata['cog_deviation'][2]:.2f})")
    print(f"Batas Deviasi: X={container.cog_limit_x:.1f}, Y={container.cog_limit_y:.1f}, Z={container.cog_limit_z:.1f}")
    print(f"Penalti Pusat Massa (B1+B2+B3): {metadata['penalty_cog']:.2f}")
    print(f"  B1 (X): {metadata['B1']:.2f}")
    print(f"  B2 (Y): {metadata['B2']:.2f}")
    print(f"  B3 (Z): {metadata['B3']:.2f}")
    print(f"B4 (Kapasitas Beban Total): {'✓ LULUS' if metadata['B4'] == 1 else '✗ GAGAL'}")
    print(f"B5 (Stabilitas): {'✓ LULUS' if metadata['B5'] == 1 else '✗ GAGAL'}")
    print(f"Fitness = (E - (B1+B2+B3)) × B4 × B5")
    print(f"        = ({metadata['total_volume']} - {metadata['penalty_cog']:.2f}) × {metadata['B4']} × {metadata['B5']}")
    print(f"        = {fitness:.2f}")
    print(f"\nPaket Terpasang: {placed_count}/{len(chromosome)}")

    return metadata['positions']

# =========== FUNGSI UTAMA ===========
def main():
    # =========== INISIALISASI DATA DEFAULT ===========
    print("=== INISIALISASI DATA DEFAULT ===\n")

    container = Container()
    packages = [
        Package("P1", 4, 3, 2, 8),
        Package("P2", 5, 2, 2, 12),
        Package("P3", 3, 3, 3, 10),
        Package("P4", 2, 4, 2, 5),
        Package("P5", 4, 2, 1, 3),
        Package("P6", 2, 2, 4, 7)
    ]

    print("=== KONTAINER ===")
    print(f"Dimensi: {container.length}x{container.width}x{container.height} cm")
    print(f"Volume: {container.volume} cm³")
    print(f"Berat maks: {container.max_weight} kg")
    print(f"Batas pusat massa: X={container.cog_limit_x:.1f}, Y={container.cog_limit_y:.1f}, Z={container.cog_limit_z:.1f} cm\n")

    print("=== PAKET ===")
    for p in packages:
        print(f"{p.id}: {p.original} cm, {p.weight} kg, Volume: {p.volume} cm³")

    # Buat packages dictionary untuk akses cepat
    packages_dict = {p.id: p for p in packages}

    # =========== INPUT PARAMETER DINAMIS ===========
    print("\n=== KONFIGURASI GENETIC ALGORITHM ===")

    # Input jumlah kromosom/populasi
    while True:
        try:
            population_size = int(input("Masukkan jumlah populasi (contoh: 10): "))
            if population_size > 0:
                break
            else:
                print("Jumlah populasi harus > 0!")
        except ValueError:
            print("Masukkan angka yang valid!")

    # Input jumlah generasi
    while True:
        try:
            generations = int(input("Masukkan jumlah generasi (contoh: 5): "))
            if generations > 0:
                break
            else:
                print("Jumlah generasi harus > 0!")
        except ValueError:
            print("Masukkan angka yang valid!")

    # Input crossover rate
    while True:
        try:
            crossover_rate = float(input("Masukkan crossover rate (0-1, contoh: 0.8): "))
            if 0 <= crossover_rate <= 1:
                break
            else:
                print("Crossover rate harus antara 0 dan 1!")
        except ValueError:
            print("Masukkan angka yang valid!")

    # Input mutation rate
    while True:
        try:
            mutation_rate = float(input("Masukkan mutation rate (0-1, contoh: 0.1): "))
            if 0 <= mutation_rate <= 1:
                break
            else:
                print("Mutation rate harus antara 0 dan 1!")
        except ValueError:
            print("Masukkan angka yang valid!")

    print(f"\nKonfigurasi GA:")
    print(f"  - Populasi: {population_size}")
    print(f"  - Generasi: {generations}")
    print(f"  - Crossover rate: {crossover_rate}")
    print(f"  - Mutation rate: {mutation_rate}")

    # Inisialisasi populasi awal
    print("\n=== INISIALISASI POPULASI AWAL ===")
    population = []

    # Generate populasi awal secara acak
    for i in range(population_size):
        chromosome = create_chromosome(list(packages_dict.values()))
        population.append(chromosome)
        print(f"K{i+1}: {chromosome}")

    # Inisialisasi Genetic Algorithm
    ga = GeneticAlgorithm(
        population_size=population_size,
        crossover_rate=crossover_rate,
        mutation_rate=mutation_rate
    )

    # Jalankan algoritma genetik
    print("\n" + "="*60)
    print("MEMULAI ALGORITMA GENETIK")
    print("="*60)

    random.seed(42)  # Untuk reproducibility

    all_results = []
    current_pop = population.copy()
    generation_details = []

    for gen in range(generations):
        print(f"\n{'='*60}")
        print(f"GENERASI {gen + 1}")
        print(f"{'='*60}")

        # 1. EVALUASI fitness populasi saat ini
        fitness_scores = []
        chrom_metadata = []

        print(f"\n>> Evaluasi Populasi ({len(current_pop)} individu):")
        print(f"   {'='*40}")
        for i, chrom in enumerate(current_pop):
            fitness, metadata = bottom_left_fill_with_fitness(chrom, container, packages_dict)
            fitness_scores.append(fitness)
            chrom_metadata.append(metadata)

            placed_count = sum(1 for p in metadata['positions'] if p['placed'])
            status_b4 = '✓' if metadata['B4'] == 1 else '✗'
            status_b5 = '✓' if metadata['B5'] == 1 else '✗'
            print(f"   K{i+1}: Fitness={fitness:8.2f} | VU={metadata['volume_utilization']:5.2f}% | "
                  f"Terpasang={placed_count:2}/{len(chrom)} | B4={status_b4} B5={status_b5}")

        print(f"   {'='*40}")
        print(f"   Fitness Range: {min(fitness_scores):.2f} - {max(fitness_scores):.2f}")
        print(f"   Fitness Avg: {np.mean(fitness_scores):.2f}")

        # Simpan hasil generasi ini
        for i, (chrom, fitness, metadata) in enumerate(zip(current_pop, fitness_scores, chrom_metadata)):
            all_results.append({
                'Generation': gen + 1,
                'Chromosome': f"K{i+1}",
                'Fitness': fitness,
                'VolumeUtil': metadata['volume_utilization'],
                'B4': metadata['B4'],
                'B5': metadata['B5'],
                'Placed': sum(1 for p in metadata['positions'] if p['placed']),
                'Sequence': chrom.copy()
            })

        if gen < generations - 1:
            # 2. GENERATE offspring dengan detail
            print(f"\n>> MEMBUAT OFFSPRING UNTUK GENERASI {gen + 2}:")
            offspring, details = ga.generate_offspring_with_details(current_pop, fitness_scores)

            # 3. EVALUASI fitness offspring
            offspring_fitness = []
            for chrom in offspring:
                fitness, _ = bottom_left_fill_with_fitness(chrom, container, packages_dict)
                offspring_fitness.append(fitness)

            # 4. GABUNGKAN populasi lama + offspring = 2N
            combined_population = current_pop + offspring
            combined_fitness = fitness_scores + offspring_fitness

            print(f"\n>> POPULASI SETELAH DIGABUNG: {len(combined_population)} individu")
            print(f"   Fitness Range: {min(combined_fitness):.2f} - {max(combined_fitness):.2f}")

            # 5. SELEKSI: ambil N terbaik dari 2N (TANPA ELITISM)
            # PERBAIKAN: Hapus argumen ketiga yang tidak diperlukan
            current_pop, selected_indices = ga.select_best(
                combined_population,
                combined_fitness
            )

            print(f"\n>> HASIL SELEKSI {population_size} INDIVIDU TERBAIK:")
            print(f"   {'='*40}")
            for i, idx in enumerate(selected_indices):
                status = "Lama" if idx < len(current_pop) else "Baru"
                print(f"   Terpilih {i+1}: Index {idx} ({status}) - Fitness: {combined_fitness[idx]:.2f}")
            print(f"   {'='*40}")

            # Simpan detail generasi
            generation_details.append({
                'generation': gen + 1,
                'population_fitness': fitness_scores.copy(),
                'offspring_fitness': offspring_fitness.copy(),
                'crossover_details': details['crossover_details'],
                'mutation_details': details['mutation_details'],
                'selected_indices': selected_indices.tolist()
            })

    # Tampilkan hasil akhir
    print("\n" + "="*60)
    print("HASIL AKHIR")
    print("="*60)

    # Cari solusi terbaik
    best_result = max(all_results, key=lambda x: x['Fitness'])

    print(f"\nSOLUSI TERBAIK:")
    print(f"Generasi: {best_result['Generation']}")
    print(f"Kromosom: {best_result['Chromosome']}")
    print(f"Fitness: {best_result['Fitness']:.2f}")
    print(f"Volume Utilization: {best_result['VolumeUtil']:.2f}%")
    print(f"B4 (Kapasitas Beban): {'LULUS' if best_result['B4'] == 1 else 'GAGAL'}")
    print(f"B5 (Stabilitas): {'LULUS' if best_result['B5'] == 1 else 'GAGAL'}")
    print(f"Paket Terpasang: {best_result['Placed']}/{len(packages)}")

    # Analisis detail kromosom terbaik
    print("\n" + "="*60)
    print("ANALISIS DETAIL KROMOSOM TERBAIK")
    print("="*60)

    # Cari kromosom terbaik di populasi
    best_chromosome = None
    for result in all_results:
        if result['Fitness'] == best_result['Fitness']:
            best_chromosome = result['Sequence']
            break

    if best_chromosome:
        best_positions = analyze_chromosome(best_chromosome, container, packages_dict)

    # Tampilkan tabel hasil
    print(f"\n{'='*60}")
    print("TABEL HASIL LENGKAP")
    print(f"{'='*60}")

    # Buat DataFrame dari hasil
    df_results = pd.DataFrame([{
        'Generasi': r['Generation'],
        'Kromosom': r['Chromosome'],
        'Fitness': f"{r['Fitness']:.2f}",
        'VU%': f"{r['VolumeUtil']:.2f}",
        'B4': '✓' if r['B4'] == 1 else '✗',
        'B5': '✓' if r['B5'] == 1 else '✗',
        'Terpasang': r['Placed']
    } for r in all_results])

    print(df_results.to_string(index=False))

    # Visualisasi kromosom terbaik
    print(f"\n{'='*60}")
    print("VISUALISASI SOLUSI TERBAIK")
    print(f"{'='*60}")

    if best_chromosome:
        fig = visualize_packing(best_positions,
                              (container.length, container.width, container.height),
                              f"Solusi Terbaik - Fitness: {best_result['Fitness']:.2f}")
        # Perbaikan: menggunakan renderer browser
        try:
            fig.show(renderer="browser")
        except:
            # Jika gagal, simpan sebagai HTML
            fig.write_html("packing_visualization.html")
            print("\nVisualisasi disimpan ke 'packing_visualization.html'")
            import webbrowser
            webbrowser.open("packing_visualization.html")

    # Export hasil ke CSV
    df_export = pd.DataFrame([{
        'Generation': r['Generation'],
        'Chromosome': r['Chromosome'],
        'Fitness': r['Fitness'],
        'VolumeUtilization': r['VolumeUtil'],
        'B4': r['B4'],
        'B5': r['B5'],
        'PlacedPackages': r['Placed'],
        'Sequence': str(r['Sequence'])
    } for r in all_results])

    df_export.to_csv('bin_packing_results_final.csv', index=False)
    print("\nHasil telah diexport ke 'bin_packing_results_final.csv'")

# Jalankan program
if __name__ == "__main__":
    main()

=== INISIALISASI DATA DEFAULT ===

=== KONTAINER ===
Dimensi: 6x5x5 cm
Volume: 150 cm³
Berat maks: 50 kg
Batas pusat massa: X=0.6, Y=0.5, Z=0.8 cm

=== PAKET ===
P1: (4, 3, 2) cm, 8 kg, Volume: 24 cm³
P2: (5, 2, 2) cm, 12 kg, Volume: 20 cm³
P3: (3, 3, 3) cm, 10 kg, Volume: 27 cm³
P4: (2, 4, 2) cm, 5 kg, Volume: 16 cm³
P5: (4, 2, 1) cm, 3 kg, Volume: 8 cm³
P6: (2, 2, 4) cm, 7 kg, Volume: 16 cm³

=== KONFIGURASI GENETIC ALGORITHM ===

Konfigurasi GA:
  - Populasi: 10
  - Generasi: 3
  - Crossover rate: 0.5
  - Mutation rate: 0.5

=== INISIALISASI POPULASI AWAL ===
K1: [('P6', 1), ('P3', 2), ('P1', 6), ('P4', 2), ('P5', 6), ('P2', 4)]
K2: [('P6', 5), ('P3', 3), ('P2', 5), ('P4', 1), ('P1', 6), ('P5', 6)]
K3: [('P6', 1), ('P2', 3), ('P4', 4), ('P3', 2), ('P5', 4), ('P1', 1)]
K4: [('P4', 6), ('P1', 3), ('P5', 6), ('P2', 5), ('P3', 5), ('P6', 2)]
K5: [('P4', 5), ('P1', 3), ('P5', 4), ('P6', 1), ('P3', 1), ('P2', 3)]
K6: [('P5', 1), ('P6', 6), ('P4', 4), ('P1', 1), ('P2', 5), ('P3', 2)]
K7: [